In [1]:
# Show GPU info
try:
    import torch
    if torch.cuda.is_available():
        idx = torch.cuda.current_device()
        print('GPU:', torch.cuda.get_device_name(idx))
        print('CUDA:', torch.version.cuda)
        print('VRAM (GB):', round(torch.cuda.get_device_properties(idx).total_memory / 1e9, 2))
    else:
        print('GPU: not available')
except Exception as e:
    print('GPU check failed:', e)

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
CUDA: 12.8
VRAM (GB): 101.97


In [2]:
%%capture
import os, re
is_colab = "COLAB_" in "".join(os.environ.keys())
if not is_colab:
    # Local: keep it simple; install the basics.
    %pip -q install unsloth transformers datasets accelerate bitsandbytes trl peft pillow scikit-learn pandas ipywidgets
else:
    # Colab: follow Unsloth's recommended install pattern (more reliable).
    import torch
    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {"2.10":"0.0.34","2.9":"0.0.33.post1","2.8":"0.0.32.post2"}.get(v, "0.0.34")
    !pip -q install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip -q install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip -q install "transformers==4.56.2"
    !pip -q install --no-deps "trl==0.22.2"
    !pip -q install "pillow<11.0"
    !pip -q install scikit-learn pandas ipywidgets

In [6]:
import os
from pathlib import Path

# Colab detection
IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ---- Data location knobs ----
DRIVE_FOLDER = os.environ.get('RX_DRIVE_FOLDER', '/content/drive/MyDrive/Internship/RX_BASE_Finetune')
ZIP_FILENAME = os.environ.get('RX_ZIP_FILENAME', 'RX_BASE_Imagenes.zip')
IMAGES_DIR = Path(os.environ.get('RX_IMAGES_DIR', '/content/images')).expanduser()
TEST_JSON = Path(os.environ.get('RX_TEST_JSON', f'{DRIVE_FOLDER}/test_data.json')).expanduser()
MODELS_DIR = Path(f'{DRIVE_FOLDER}/Models')

# Evaluation knobs
EVAL_SIZE = int(os.environ.get('RX_EVAL_SIZE', '0'))  # 0 = all test examples
EVAL_TEMPERATURE = float(os.environ.get('RX_EVAL_TEMPERATURE', '0.05'))
ANSWER_STYLE = 'label_only'  # Fixed for evaluation

print('IN_COLAB:', IN_COLAB)
print('DRIVE_FOLDER:', DRIVE_FOLDER)
print('MODELS_DIR:', MODELS_DIR)
print('IMAGES_DIR:', IMAGES_DIR)
print('TEST_JSON:', TEST_JSON)
print('EVAL_SIZE:', EVAL_SIZE if EVAL_SIZE > 0 else 'all')

# ---- Mount Drive and prep data ----
if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    
    if not MODELS_DIR.exists():
        raise FileNotFoundError(f"Models folder not found: {MODELS_DIR}")
    
    # Unzip images if needed
    zip_path = f"{DRIVE_FOLDER}/{ZIP_FILENAME}"
    if (not IMAGES_DIR.exists()) or (IMAGES_DIR.exists() and len(list(IMAGES_DIR.glob('*.jpg'))) < 100):
        print('📦 Extracting images...')
        !unzip -q "{zip_path}" -d /content/
        !mkdir -p /content/images
        !mv /content/Imagenes/* /content/images/ 2>/dev/null || true
        print('✅ Extracted images into', IMAGES_DIR)
    else:
        print('✅ Images already present')

IN_COLAB: True
DRIVE_FOLDER: /content/drive/MyDrive/Internship/RX_BASE_Finetune
MODELS_DIR: /content/drive/MyDrive/Internship/RX_BASE_Finetune/Models
IMAGES_DIR: /content/images
TEST_JSON: /content/drive/MyDrive/Internship/RX_BASE_Finetune/test_data.json
EVAL_SIZE: all
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Images already present


In [4]:
# List available models and let user select one
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

# Find all model directories (rx_lora_*)
model_dirs = sorted([d for d in MODELS_DIR.glob('rx_lora_*') if d.is_dir()], reverse=True)

if not model_dirs:
    raise FileNotFoundError(f"No models found in {MODELS_DIR}. Train a model first.")

print(f"Found {len(model_dirs)} trained models:")
for i, d in enumerate(model_dirs, 1):
    print(f"  {i}. {d.name}")

# Create dropdown widget
model_options = [(d.name, str(d)) for d in model_dirs]
model_selector = widgets.Dropdown(
    options=model_options,
    description='Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px')
)

print("\nSelect a model to evaluate:")
display(model_selector)

# Store selection (will be used in next cells)
SELECTED_MODEL_PATH = None

def on_model_selected(change):
    global SELECTED_MODEL_PATH
    SELECTED_MODEL_PATH = Path(change['new'])
    print(f"Selected: {SELECTED_MODEL_PATH.name}")

model_selector.observe(on_model_selected, names='value')

# Auto-select first model
SELECTED_MODEL_PATH = Path(model_options[0][1])
print(f"\nDefault selection: {SELECTED_MODEL_PATH.name}")

Found 2 trained models:
  1. rx_lora_20260226_004614
  2. rx_lora_20260225_171147

Select a model to evaluate:


Dropdown(description='Model:', layout=Layout(width='600px'), options=(('rx_lora_20260226_004614', '/content/dr…


Default selection: rx_lora_20260226_004614


In [7]:
# Load test data
import json
from pathlib import Path

with open(TEST_JSON, 'r', encoding='utf-8') as f:
    test_raw = json.load(f)

print('Test examples:', len(test_raw))
print('First keys:', list(test_raw[0].keys()))

# Helper to extract label
def _to_label_only(text: str) -> str:
    rock = text.split('.', 1)[0].strip()
    return rock if rock else text.strip()

# Convert to messages format
def convert_to_messages(ex, images_dir: Path):
    image_name = Path(str(ex['image'])).name
    image_path = images_dir / image_name
    
    convs = ex['conversations']
    user_text = str(convs[0].get('value', '')).replace('<image>\n', '').replace('<image>', '').strip()
    user_text = user_text + " Reply with only the rock type name."
    assistant_text = _to_label_only(str(convs[1].get('value', '')).strip())
    
    return {
        'messages': [
            {
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': str(image_path)},
                    {'type': 'text', 'text': user_text},
                ],
            },
            {
                'role': 'assistant',
                'content': [
                    {'type': 'text', 'text': assistant_text},
                ],
            },
        ]
    }

test_data = test_raw if EVAL_SIZE == 0 else test_raw[:min(EVAL_SIZE, len(test_raw))]
test_dataset = [convert_to_messages(ex, IMAGES_DIR) for ex in test_data]
print('Converted test examples:', len(test_dataset))

Test examples: 1120
First keys: ['image', 'conversations']
Converted test examples: 1120


In [8]:
# Load base model + LoRA adapters
import torch
from unsloth import FastVisionModel

print(f"Loading model from: {SELECTED_MODEL_PATH}")

# Read adapter config to determine base model
import json
adapter_config_path = SELECTED_MODEL_PATH / 'adapter_config.json'
with open(adapter_config_path, 'r') as f:
    adapter_config = json.load(f)
    base_model_name = adapter_config.get('base_model_name_or_path', 'unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit')

print(f"Base model: {base_model_name}")

LOAD_IN_4BIT = ('4bit' in base_model_name) or ('bnb-4bit' in base_model_name)

# Load base model
model, tokenizer = FastVisionModel.from_pretrained(
    base_model_name,
    load_in_4bit=LOAD_IN_4BIT,
    use_gradient_checkpointing='unsloth',
)

# Load LoRA adapters
from peft import PeftModel
print(f"Loading LoRA adapters...")
model = PeftModel.from_pretrained(model, str(SELECTED_MODEL_PATH))

# Set to inference mode
FastVisionModel.for_inference(model)

print('✅ Model loaded and ready for inference')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model from: /content/drive/MyDrive/Internship/RX_BASE_Finetune/Models/rx_lora_20260226_004614
Base model: unsloth/llama-3.2-11b-vision-instruct-bnb-4bit
==((====))==  Unsloth 2026.2.1: Fast Mllama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.18G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

Loading LoRA adapters...
✅ Model loaded and ready for inference


In [9]:
# Run evaluation on test set
import re
from typing import List
import pandas as pd
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from tqdm.auto import tqdm

def _extract_label(text: str) -> str:
    if text is None:
        return ""
    t = str(text).strip()
    if not t:
        return ""
    t = t.splitlines()[0].strip()
    t = t.split(".", 1)[0].strip()
    t = re.sub(r"[^A-Za-z0-9 _\-]+", "", t).strip()
    return t

print(f"Evaluating on {len(test_dataset)} test examples...")

y_true: List[str] = []
y_pred: List[str] = []
unknown_preds = 0

for item in tqdm(test_dataset, desc='Evaluating'):
    img_path = item['messages'][0]['content'][0]['image']
    prompt = item['messages'][0]['content'][1]['text']
    true_text = item['messages'][1]['content'][0]['text']
    
    img = Image.open(img_path).convert('RGB')
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image'},
                {'type': 'text', 'text': prompt},
            ],
        }
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(img, input_text, add_special_tokens=False, return_tensors='pt').to('cuda')
    input_len = inputs['input_ids'].shape[-1]
    
    out = model.generate(
        **inputs,
        max_new_tokens=32,
        temperature=EVAL_TEMPERATURE,
        use_cache=False,
        min_p=0.1,
    )
    new_tokens = out[0][input_len:]
    pred_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    yt = _extract_label(true_text)
    yp = _extract_label(pred_text)
    if not yp:
        unknown_preds += 1
        yp = "<EMPTY>"
    y_true.append(yt)
    y_pred.append(yp)

labels_true = sorted(set(y_true))
extra_labels = sorted(set(y_pred) - set(labels_true))
labels_cm = labels_true + extra_labels

acc = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, labels=labels_true, average='macro', zero_division=0)
f1_weighted = f1_score(y_true, y_pred, labels=labels_true, average='weighted', zero_division=0)
prec_macro = precision_score(y_true, y_pred, labels=labels_true, average='macro', zero_division=0)
rec_macro = recall_score(y_true, y_pred, labels=labels_true, average='macro', zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=labels_cm)

print('\n' + '='*60)
print(f'EVALUATION RESULTS: {SELECTED_MODEL_PATH.name}')
print('='*60)
print(f'Test examples: {len(test_dataset)}')
print(f'Accuracy: {round(acc, 4)}')
print(f'F1 macro: {round(f1_macro, 4)} | F1 weighted: {round(f1_weighted, 4)}')
print(f'Precision macro: {round(prec_macro, 4)} | Recall macro: {round(rec_macro, 4)}')
print(f'Empty/unknown predictions: {unknown_preds}')
print('='*60)

print('\nPer-class classification report:')
print(classification_report(y_true, y_pred, labels=labels_true, zero_division=0))

print('\nConfusion Matrix:')
cm_df = pd.DataFrame(
    cm,
    index=[f"true:{l}" for l in labels_cm],
    columns=[f"pred:{l}" for l in labels_cm],
)
cm_df

Evaluating on 1120 test examples...


Evaluating:   0%|          | 0/1120 [00:00<?, ?it/s]


EVALUATION RESULTS: rx_lora_20260226_004614
Test examples: 1120
Accuracy: 0.9116
F1 macro: 0.9112 | F1 weighted: 0.9112
Precision macro: 0.9122 | Recall macro: 0.9116
Empty/unknown predictions: 0

Per-class classification report:
              precision    recall  f1-score   support

    Andesite       0.89      0.91      0.90        80
      Basalt       0.89      0.93      0.91        80
     Diorite       0.86      0.88      0.87        80
      Gabbro       0.89      0.93      0.91        80
      Gneiss       0.82      0.82      0.82        80
     Granite       0.96      0.80      0.87        80
   Limestone       0.97      0.95      0.96        80
      Marble       0.94      0.97      0.96        80
  Peridotite       0.94      0.97      0.96        80
    Phyllite       0.92      0.91      0.92        80
    Rhyolite       0.88      0.91      0.90        80
   Sandstone       0.97      0.97      0.97        80
      Schist       0.88      0.84      0.86        80
       Shale

,pred:Andesite,pred:Basalt,pred:Diorite,pred:Gabbro,pred:Gneiss,pred:Granite,pred:Limestone,pred:Marble,pred:Peridotite,pred:Phyllite,pred:Rhyolite,pred:Sandstone,pred:Schist,pred:Shale
true:Andesite,73,5,0,0,0,0,0,0,1,0,1,0,0,0
true:Basalt,5,74,0,0,0,0,0,0,0,0,1,0,0,0
true:Diorite,0,0,70,6,2,2,0,0,0,0,0,0,0,0
true:Gabbro,0,0,2,74,1,0,0,0,3,0,0,0,0,0
true:Gneiss,0,0,2,1,66,1,1,1,0,0,0,1,7,0
true:Granite,0,0,5,1,2,64,0,3,0,1,3,1,0,0
true:Limestone,0,0,0,0,0,0,76,0,0,0,1,0,1,2
true:Marble,0,0,0,0,0,0,0,78,1,0,1,0,0,0
true:Peridotite,0,1,0,1,0,0,0,0,78,0,0,0,0,0
true:Phyllite,0,0,2,0,0,0,0,0,0,73,1,0,1,3


In [10]:
# Interactive inference widget
import ipywidgets as widgets
from IPython.display import display, Image as IPImage, clear_output
from PIL import Image
import random

# Pick random test images for demo
demo_indices = random.sample(range(len(test_dataset)), min(100, len(test_dataset)))
demo_items = [test_dataset[i] for i in demo_indices]

current_idx = [0]  # Store in list so we can modify in closure
output_area = widgets.Output()

def show_inference(idx):
    with output_area:
        clear_output(wait=True)
        item = demo_items[idx]
        img_path = item['messages'][0]['content'][0]['image']
        prompt = item['messages'][0]['content'][1]['text']
        true_text = item['messages'][1]['content'][0]['text']
        
        # Display image
        display(IPImage(filename=img_path, width=400))
        print(f"\nPrompt: {prompt}")
        print(f"Ground truth: {true_text}")
        print("\nPrediction: ", end='', flush=True)
        
        # Generate prediction
        img = Image.open(img_path).convert('RGB')
        messages = [
            {
                'role': 'user',
                'content': [
                    {'type': 'image'},
                    {'type': 'text', 'text': prompt},
                ],
            }
        ]
        input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
        inputs = tokenizer(img, input_text, add_special_tokens=False, return_tensors='pt').to('cuda')
        input_len = inputs['input_ids'].shape[-1]
        
        out = model.generate(
            **inputs,
            max_new_tokens=32,
            temperature=0.1,
            use_cache=False,
            min_p=0.1,
        )
        new_tokens = out[0][input_len:]
        pred_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        
        pred_label = _extract_label(pred_text)
        true_label = _extract_label(true_text)
        match = "✅" if pred_label.lower() == true_label.lower() else "❌"
        print(f"{pred_text} {match}")

def on_next_clicked(b):
    current_idx[0] = (current_idx[0] + 1) % len(demo_items)
    show_inference(current_idx[0])

def on_prev_clicked(b):
    current_idx[0] = (current_idx[0] - 1) % len(demo_items)
    show_inference(current_idx[0])

next_btn = widgets.Button(description='Next ▶')
prev_btn = widgets.Button(description='◀ Previous')
next_btn.on_click(on_next_clicked)
prev_btn.on_click(on_prev_clicked)

controls = widgets.HBox([prev_btn, next_btn])

print("Interactive Inference Demo")
print("==========================\n")
display(controls)
display(output_area)

# Show first example
show_inference(0)

Interactive Inference Demo



Output()

In [ ]:
# Interactive inference: upload your own image or pick from test set
import io
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image

# Build selectable test images (name -> full path)
test_image_options = []
for item in test_dataset:
    p = item['messages'][0]['content'][0]['image']
    test_image_options.append((Path(p).name, p))

# Keep first occurrence for duplicate names
seen = set()
dedup_options = []
for name, path in test_image_options:
    if name not in seen:
        seen.add(name)
        dedup_options.append((name, path))

source_selector = widgets.ToggleButtons(
    options=['Select from test set', 'Upload image'],
    description='Source:',
    style={'description_width': 'initial'}
)

test_dropdown = widgets.Dropdown(
    options=dedup_options[:500],
    description='Test image:',
    layout=widgets.Layout(width='650px'),
    style={'description_width': 'initial'}
)

uploader = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Upload image',
)

prompt_box = widgets.Textarea(
    value='Classify this petrographic thin section image. What type of rock is this? Reply with only the rock type name.',
    description='Prompt:',
    layout=widgets.Layout(width='900px', height='80px'),
    style={'description_width': 'initial'}
)

run_btn = widgets.Button(description='Run inference', button_style='primary')
output_box = widgets.Output()

def _extract_uploaded_bytes(upload_value):
    # ipywidgets can return dict-like or tuple-like structures depending on version
    if upload_value is None:
        return None, None
    if isinstance(upload_value, dict):
        if len(upload_value) == 0:
            return None, None
        first_key = next(iter(upload_value))
        file_obj = upload_value[first_key]
        content = file_obj.get('content', None)
        return content, first_key
    try:
        if len(upload_value) == 0:
            return None, None
        first = upload_value[0]
        name = first.get('name', 'uploaded_image') if isinstance(first, dict) else getattr(first, 'name', 'uploaded_image')
        content = first.get('content', None) if isinstance(first, dict) else getattr(first, 'content', None)
        return content, name
    except Exception:
        return None, None

def _predict_single_image(pil_img, prompt_text):
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image'},
                {'type': 'text', 'text': prompt_text},
            ],
        }
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(pil_img, input_text, add_special_tokens=False, return_tensors='pt').to('cuda')
    input_len = inputs['input_ids'].shape[-1]

    out = model.generate(
        **inputs,
        max_new_tokens=32,
        temperature=EVAL_TEMPERATURE,
        use_cache=False,
        min_p=0.1,
    )
    new_tokens = out[0][input_len:]
    pred_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return pred_text

def on_source_change(change):
    using_upload = change['new'] == 'Upload image'
    uploader.layout.display = '' if using_upload else 'none'
    test_dropdown.layout.display = 'none' if using_upload else ''

source_selector.observe(on_source_change, names='value')
on_source_change({'new': source_selector.value})

def on_run_clicked(_):
    with output_box:
        clear_output(wait=True)
        prompt_text = prompt_box.value.strip()
        if not prompt_text:
            print('Please enter a prompt.')
            return

        image_name = None
        pil_img = None

        if source_selector.value == 'Upload image':
            content, image_name = _extract_uploaded_bytes(uploader.value)
            if content is None:
                print('Please upload an image first.')
                return
            pil_img = Image.open(io.BytesIO(content)).convert('RGB')
        else:
            chosen_path = test_dropdown.value
            if not chosen_path:
                print('Please select a test image.')
                return
            image_name = Path(chosen_path).name
            pil_img = Image.open(chosen_path).convert('RGB')

        print(f'Image: {image_name}')
        display(pil_img.resize((min(512, pil_img.width), int(pil_img.height * min(512, pil_img.width) / pil_img.width))))
        print('\nPrediction: ', end='')
        pred_text = _predict_single_image(pil_img, prompt_text)
        pred_label = _extract_label(pred_text)
        print(pred_text)
        if pred_label:
            print('Parsed label:', pred_label)

run_btn.on_click(on_run_clicked)

print('Custom Interactive Inference')
print('Pick a test image OR upload your own image, then click Run inference.\n')
display(source_selector)
display(test_dropdown)
display(uploader)
display(prompt_box)
display(run_btn)
display(output_box)

Custom Interactive Inference
Pick a test image OR upload your own image, then click Run inference.



ToggleButtons(description='Source:', options=('Select from test set', 'Upload image'), style=ToggleButtonsStyl…

Dropdown(description='Test image:', layout=Layout(display='', width='650px'), options=(('101_00255.jpg', '/con…

FileUpload(value={}, accept='image/*', description='Upload image', layout=Layout(display='none'))

Textarea(value='Classify this petrographic thin section image. What type of rock is this? Reply with only the …

Button(button_style='primary', description='Run inference', style=ButtonStyle())

Output()

In [ ]:
# Save evaluation results to Drive
import json
from datetime import datetime

if IN_COLAB:
    results = {
        'model_name': SELECTED_MODEL_PATH.name,
        'model_path': str(SELECTED_MODEL_PATH),
        'timestamp': datetime.now().isoformat(),
        'test_size': len(test_dataset),
        'metrics': {
            'accuracy': round(acc, 4),
            'f1_macro': round(f1_macro, 4),
            'f1_weighted': round(f1_weighted, 4),
            'precision_macro': round(prec_macro, 4),
            'recall_macro': round(rec_macro, 4),
            'unknown_predictions': unknown_preds,
        },
        'per_class_report': classification_report(y_true, y_pred, labels=labels_true, zero_division=0, output_dict=True),
    }
    
    results_file = SELECTED_MODEL_PATH / 'evaluation_results.json'
    with open(results_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2)
    
    print(f"✅ Evaluation results saved to: {results_file}")
else:
    print("Not on Colab; skipping save to Drive.")